In [25]:
import pandas as pd
import os
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture

In [26]:
# Select candidates 

person1 = 'Martins'
person2 = 'Gouveia_Melo'

In [27]:
# Load visual pkl for debate with person 1 and person 2 OR all debates with person 1 OR all debates
pklfiles = []

if not person1:
    for file in os.listdir('Project_Features'):
        if file.endswith('visual.pkl'):
            pklfiles.append(file)
elif not person2:
    for file in os.listdir('Project_Features'):
        if file.endswith('visual.pkl') and person1 in file:
            pklfiles.append(file)
else:
    for file in os.listdir('Project_Features'):
        if file.endswith('visual.pkl') and person1 in file and person2 in file:
            pklfiles.append(file)


data = []

for video in pklfiles:
    print(f"Loading: {video}")
    df = pd.read_pickle(os.path.join('Project_Features', video))
    data.append(df)

data = pd.concat(data, ignore_index=True)

data['Video_Name'] = data['Frame'].str.extract(r'Frames/([^/]+)/')
data['Frame_Number'] = data['Frame'].str.extract(r'frame_(\d+)\.jpg').astype(int)
data = data.sort_values(by=['Video_Name', 'Frame_Number']).reset_index(drop=True)


Loading: Martins_vs_Gouveia_Melo_November_23_visual.pkl


In [28]:
master_data = []

for index, row in data.iterrows():
    frame_id = row['Frame']
    frame_number = row['Frame_Number']
    faces = row['Fer']
    poses = row['Poses'] # This contains the body bbox and the 17 keypoints
    
    if isinstance(faces, list) and len(faces) > 0 and len(faces)<=3:
        valid_faces = [f for f in faces if isinstance(f, dict) and 'bbox' in f]
        people_count = len(valid_faces)
        
        for i, face in enumerate(valid_faces):
            # 1. Face Coordinates
            f_box = face['bbox'] # [x1, y1, x2, y2]
            f_width = f_box[2] - f_box[0]
            f_height = f_box[3] - f_box[1]
            
            f_x_center = f_box[0] + (f_width / 2)
            f_y_center = f_box[1] + (f_height / 2) # Get Y center too!
            face_area = f_width * f_height
            
            top_emotion = face.get('top_emotion')
            probability = face['probabilities'].get(top_emotion) if top_emotion and 'probabilities' in face else None
            landmarks = face.get('landmarks')
            
            # 2. MATCH THE BODY TO THE FACE
            matched_body_bbox = None
            matched_pose_keypoints = None
            
            if isinstance(poses, list):
                for person_body in poses:
                    if isinstance(person_body, dict)and len(person_body) > 0:
                        b_box = person_body['bbox'] 
                        
                        # is the center of the face inside this body's bounding box?
                        #note that b_box[3] > bbox[1] because y axis goes down
                        if (b_box[0] <= f_x_center <= b_box[2]) and (b_box[1] <= f_y_center <= b_box[3]):
                            matched_body_bbox = b_box
                            matched_pose_keypoints = person_body.get('pose') # The 17x3 matrix
                            break 
            
            # 3. Append everything safely
            master_data.append({
                'Frame': frame_id,
                'Frame_Number': frame_number,
                'People_Count': people_count,
                'Person_Index': i,
                'Face_X_Center': f_x_center,
                'Face_Area': face_area,
                'Face_BBox': f_box,
                'Body_BBox': matched_body_bbox,         
                'Top_Emotion': top_emotion,
                'Emotion_Prob': probability,
                'Landmarks': landmarks,
                'Pose_Keypoints': matched_pose_keypoints 
            })
            
    else:
        # Handle empty frames
        master_data.append({
            'Frame': frame_id, 'Frame_Number': frame_number, 'People_Count': 0, 'Person_Index': None,
            'Face_X_Center': None, 'Face_Area': None, 'Face_BBox': None,
            'Body_BBox': None, 'Top_Emotion': None, 'Emotion_Prob': None,
            'Landmarks': None, 'Pose_Keypoints': None
        })

df_people = pd.DataFrame(master_data)
print(f"Important data (df shape): {df_master.shape}")

Important data (df shape): (3826, 12)


In [29]:
#because our data frame is separated into one person (instead of 1 frame=1row)
#we have to group the rows that correspond to the same frame to plot the timeline

grouped_rows = df_people.groupby('Frame_Number')
people_in_frame= grouped_rows['People_Count'].max() #in case i have the same frame detecting 1 or 2 faces --> keep2 
df_frames = people_in_frame.reset_index()

# 
counts = df_frames['People_Count'].value_counts().sort_index()
print("Screen time by number of people detected:")
for n_people, n_frames in counts.items():
    minuts = n_frames//60
    secs = n_frames%60
    print(f"  {n_people} people: {n_frames} frames ({minuts}min {secs}s)")



Screen time by number of people detected:
  0 people: 2 frames (0min 2s)
  1 people: 989 frames (16min 29s)
  2 people: 901 frames (15min 1s)
  3 people: 156 frames (2min 36s)


In [ ]:
#only call this when you only have 1 person at the shot
def classify_single_people(row):
    pose = row['Pose_Keypoints']
    if pose is not None:

        #IM HEREEEEE    
        return
